In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
import rpy2.situation
for row in rpy2.situation.iter_info():
    print(row)
from rpy2.robjects import r
from rpy2.robjects.packages import importr
%matplotlib qt
from local_functions import pix_to_deg,read_data,calculate_bf,calculate_bf_nans,plot_bfs,shrink_axes,make_subplots, inset_axes
from statsmodels.formula.api import probit
from statsmodels.formula.api import ols
from scipy.stats import linregress

import rpy2.robjects as ro
ro.r('.libPaths(c(.libPaths(), "/Library/Frameworks/R.framework/Versions/4.6/Resources/library"))')
bf_package=importr('BayesFactor')

# A4 dimensions in inches
a4_width = 8.27 
a4_height = 11.69 

task = 'exp3'

subj_colors = sns.color_palette("Spectral",21)
colors = sns.diverging_palette(250, 30, l=70, n=12,center='dark')



CFFI extension type
  ABI: PRESENT
  API: PRESENT
  Environment variable: RPY2_CFFI_MODE=ANY
  Loaded: CFFI_MODE.API
rpy2 version:
rpy2.rinterface: 3.6.6
Python version:
3.14.4 | packaged by Anaconda, Inc. | (main, Apr 14 2026, 17:01:42) [Clang 20.1.8 ]
Looking for R's HOME:
  Environment variable R_HOME: None
  Calling `R RHOME`: /Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/R
  Environment variable R_LIBS_USER: None
R's value for LD_LIBRARY_PATH:
  
R version:
  In the PATH: R version 4.5.3 (2026-03-11) -- "Reassured Reassurer"
  Loading R library from rpy2: OK
Additional directories to load R packages from:
  None
C extension compilation:
  include:
    ['/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/R/include']
  libraries:
    ['R', 'bz2', 'z', 'dl', 'm', 'iconv', 'icuuc', 'icui18n']
  library_dirs:
    ['/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib', '/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/R/lib', '/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib']
  extra_compile_

### read data

In [2]:
all_epochs = []
bids_dir = '/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data'
subjids = [i for i in np.arange(1,21)]
all_epochs = read_data(bids_dir,subjids,task = 'exp3')

Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/preprocessed/sub-S01_exp3_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    3500.00 ms
        0 CTF compensation matrices available
Adding metadata with 27 columns
384 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/preprocessed/sub-S02_exp3_eyetracking-preproc-epo.fif ...
Isotrak not found
    Found the data of interest:
        t =    -500.00 ...    3500.00 ms
        0 CTF compensation matrices available
Adding metadata with 27 columns
384 matching events found
No baseline correction applied
0 projection items activated
Reading /Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/preprocessed/sub-

In [3]:
gaze_error = []
beh_error = []

i = 0
condition = 'occluded'
epos = all_epochs[i].copy()
epos = epos[epos.metadata.condition==condition]
x_idx = np.where([ii=='xpos' for ii in epos.ch_names])[0][0]
epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

beh_error.append(epos.metadata.loc[(epos.metadata.condition=='occluded'),'rt'].to_numpy())


tv = epos.times.copy()
end_time = epos.metadata.loc[:,'occlusion_end_time']
end_time_idx = np.array([np.argmin(np.abs(tv-i)) for i in end_time])



gaze_error.append([epos._data[trial_i,x_idx,trial_end_idx] for trial_i,trial_end_idx in enumerate(end_time_idx)])




### make plot

In [4]:
# pulling trialwise data from each participant, making a plot per person
plt.close('all')
temp_error_transparent_all = []
temp_error_occluded_all = []
error_deg_transparent_all = []
error_deg_occluded_all = []
final_gaze_error_occluded = []
final_gaze_error_transparent = []
trialwise_gaze_spatial_error = []
trialwise_behavioral_spatial_error = []

for i in range(20):
    epos = all_epochs[i].copy()
    fig,axs = plt.subplots(2,2,num=i,figsize=(16,8))
    temp_error_transparent_all.append(epos.metadata.loc[(epos.metadata.condition=='transparent'),:].groupby('velocity').mean(numeric_only=True).rt-epos.metadata.loc[(epos.metadata.condition=='transparent'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time)
    temp_error_occluded_all.append(epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).rt-epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time)
    
    ax = axs[0][0]
    ax.plot(temp_error_transparent_all[i],'grey',alpha=0.7,label='transparent')
    ax.plot(temp_error_occluded_all[i],'red',alpha=0.7,label='occluded')
    ax.set_xlabel('velocity')
    ax.set_ylabel('temporal error (s)')
    ax.set_ylim(-0.2,1.6)
    ax.legend()
    error_deg_occluded_all.append(temp_error_occluded_all[i].index*temp_error_occluded_all[i])
    error_deg_transparent_all.append(temp_error_transparent_all[i].index*temp_error_transparent_all[i])
    
    ax = axs[0][1]
    ax.plot(error_deg_transparent_all[i],'grey',alpha=0.7,label='transparent')
    ax.plot(error_deg_occluded_all[i],'red',alpha=0.7,label='occluded')
    ax.set_xlabel('velocity')
    ax.set_ylabel('spatial error (dva)')
    ax.set_ylim(-1.3,10)
    ax.legend()
    fig.suptitle(f'sub-S{str(i+1).zfill(2)}',weight='bold')
    # eye tracking
    for condition,ax in zip(['transparent', 'occluded'],[axs[1][0],axs[1][1]]):
        epos = all_epochs[i].copy()
        epos = epos[epos.metadata.condition==condition]
        x_idx = np.where([ii=='xpos' for ii in epos.ch_names])[0][0]
        epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1
        speeds = np.sort(epos.metadata.velocity.unique())
        for speed_idx in range(len(speeds)):
            end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]
            tv = epos.times.copy()
            tv[tv>end_time]=np.nan
            trials = epos._data[epos.metadata.velocity==speeds[speed_idx],x_idx,:]
            trials[:,tv>end_time]=np.nan
            [ax.plot(tv,trial,color=colors[speed_idx],lw=0.2,alpha=1) for trial in trials]
            ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
            ax.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=15,zorder=500,markeredgecolor='k')

        ax.set_ylabel('x-gaze position (dva)')
        ax.set_xlabel('time (s) since trial start')
        ax.set_title(condition)
        ax.set_ylim(-12,12)

        # trialwise spatial errors (save out for later)
        end_times = epos.metadata['occlusion_end_time'].values
        velocities = epos.metadata['velocity'].values
        gaze_at_end = np.array([epos._data[t, x_idx, np.argmin(np.abs(epos.times - end_times[t]))] for t in range(len(epos))])
        df = epos.metadata[['velocity','direction','rt','occlusion_end_time']].copy()
        df['gaze_spatial_error'] = np.abs(epos.metadata.occluder_start_pos_deg)-gaze_at_end
        df['behavioral_spatial_error'] = (epos.metadata['rt'].values - end_times) * velocities
        df['condition'] = condition
        df['subject'] = i + 1
        trialwise_gaze_spatial_error.append(df)

    plt.tight_layout()
    plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/individual_data/trialwise_errors/sub-S{str(i+1).zfill(2)}_trialwise_errors.png')
                

trialwise_df = pd.concat(trialwise_gaze_spatial_error, ignore_index=True)

/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1103449526.py:50: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1103449526.py:50: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1103449526.py:50: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1103449526.py:50: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1103449526.py:50: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1)
/var/folders/bh/8mpq

In [5]:
print(epos.metadata.columns)

Index(['Unnamed: 0', 'participant', 'block', 'trial', 'direction', 'start_pos',
       'start_pos_deg', 'speed_idx', 'total_time', 'pix_per_refresh',
       'pix_per_sec', 'pos_pressed', 'pos_pressed_deg', 'rt', 'widthpix',
       'heightpix', 'viewdist', 'monitorwidth', 'subjid', 'condition',
       'monitor_refreshrate', 'trajectory_distance_deg', 'velocity',
       'n_frames', 'occluder_start_pos_deg', 'occlusion_start_time',
       'occlusion_end_time'],
      dtype='str')


In [5]:
transparent_output = []
occluded_output = []


for i in range(20):
    plt.close('all')
    fig,axs = plt.subplots(6,2,num=i,figsize=(7,16),sharex=True,sharey=True)

    for condition in (['transparent','occluded']):
        epos = all_epochs[i].copy()
        epos = epos[epos.metadata.condition==condition]
        epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

        subject_output = []

        for ax,speed_idx in zip(axs.flatten(),range(len(speeds))):
            end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]
            tv = epos.times.copy()
            tv[tv>end_time]=np.nan
            trials = epos._data[epos.metadata.velocity==speeds[speed_idx],x_idx,:]
            trials[:,tv>end_time]=np.nan

            mean_trials = np.nanmean(trials,axis=0)

            # append mean trajectory for this speed
            subject_output.append(mean_trials)

            if condition =='transparent':
                ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=0.4,label='transparent')
            else:
                ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=1,label='occluded')
            ax.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=8,zorder=500,markeredgecolor='k')
            ax.set_ylim(-12,12)

            ax.legend(fontsize=6,frameon=False)

        # append this subject
        if condition == 'transparent':
            transparent_output.append(subject_output)
        else:
            occluded_output.append(subject_output)

    fig.supylabel('x-gaze position (dva)')
    fig.supxlabel('time (s) since trial start')
    fig.suptitle(f'sub-S{str(i+1).zfill(2)}',weight='bold')

    plt.tight_layout()

    plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/individual_data/sub-S{str(i+1).zfill(2)}_eyetracking_transparent-vs-occluded.png')


/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2760166352.py:23: RuntimeWarning: Mean of empty slice
  mean_trials = np.nanmean(trials,axis=0)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2760166352.py:29: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=0.4,label='transparent')
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2760166352.py:23: RuntimeWarning: Mean of empty slice
  mean_trials = np.nanmean(trials,axis=0)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2760166352.py:29: RuntimeWarning: Mean of empty slice
  ax.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=2,alpha=0.4,label='transparent')
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2760166352.py:23: RuntimeWarning: Mean of empty slice
  mean_trials = np.nanmean(trials,axis=0)
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2760166352.py:29

In [6]:
# create overlay of gaze position for transparent and occluded trial types across all subjects

transparent_output = np.array(transparent_output)
occluded_output = np.array(occluded_output)

# average across subjects for each speed
transparent_ave = np.nanmean(transparent_output, axis=0)
occluded_ave = np.nanmean(occluded_output, axis=0)

# plot group averages
plt.close('all')
fig, axs = plt.subplots(6, 2, figsize=(7,16), sharex=True, sharey=True)

epos = all_epochs[0].copy()
summary_dat = epos.metadata.loc[epos.metadata.direction==1,:].groupby('velocity').mean(numeric_only=True)

bf_transparent = []
bf_occluded = []
bf_trans_occ = []

val_trans = []
val_occ = []
val_trans_occ = []

bf_times = []

for ax, speed_idx in zip(axs.flatten(), range(len(speeds))):

    # get end time separately for each speed
    end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]

    # create time vector separately for each speed
    tv = epos.times.copy()

    tv[tv>end_time] = np.nan

    # TRUE TARGET TRAJECTORY
    traj = np.linspace(summary_dat['start_pos_deg'].iloc[speed_idx],np.abs(summary_dat['start_pos_deg'].iloc[speed_idx]),int(summary_dat['n_frames'].iloc[speed_idx]))

    traj_tv = np.linspace(0,summary_dat['total_time'].iloc[speed_idx],int(summary_dat['n_frames'].iloc[speed_idx]))

    # time period during which target is occluded
    occlusion_mask_traj = (traj >= -4.5) & (traj <= 4.5)

    occlusion_start = traj_tv[occlusion_mask_traj][0]
    occlusion_end = traj_tv[occlusion_mask_traj][-1]

    # corresponding gaze samples
    occlusion_mask_gaze = ((epos.times >= occlusion_start) &(epos.times <= occlusion_end))

    gaze_time_dur_occlusion = epos.times[occlusion_mask_gaze]

    # true occluded trajectory
    true_target_occluded = np.interp(gaze_time_dur_occlusion,traj_tv,traj)

    ## BF

    # subjects × occlusion time

    # true vs. transparent
    trans_diff = (transparent_output[:, speed_idx, occlusion_mask_gaze] - true_target_occluded[None, :])

    # true vs. transparent
    occ_diff = (occluded_output[:, speed_idx, occlusion_mask_gaze] - true_target_occluded[None, :])

    # occluded vs. transparent
    trans_occ_diff = (transparent_output[:, speed_idx, occlusion_mask_gaze] - occluded_output[:, speed_idx, occlusion_mask_gaze])

    bf_transparent.append(calculate_bf_nans(trans_diff))
    bf_occluded.append(calculate_bf_nans(occ_diff))
    bf_trans_occ.append(calculate_bf_nans(trans_occ_diff))

    #use later for inidcating direction of effect
    val_trans.append(np.nanmean(trans_diff, axis=0)) #val_trans.append(trans_diff)
    val_occ.append(np.nanmean(occ_diff, axis=0)) #val_occ.append(occ_diff)
    val_trans_occ.append(np.nanmean(trans_occ_diff, axis=0))

    #BF times for plotting
    bf_times.append(gaze_time_dur_occlusion)

    ## PLOTTING

    # visible trajectory
    ax.plot(traj_tv[traj < -4.5],traj[traj < -4.5],'-',color=colors[speed_idx],lw=0.5)

    # occluded trajectory
    ax.plot(traj_tv[(traj >= -4.5) & (traj <= 4.5)],traj[(traj >= -4.5) & (traj <= 4.5)],':',color=colors[speed_idx],lw=0.5)

    # GAZE POSITION
    ax.plot(tv,transparent_ave[speed_idx, :],color=colors[speed_idx],lw=2,alpha=0.4,label='transparent')

    ax.plot(tv,occluded_ave[speed_idx, :],color=colors[speed_idx],lw=2,alpha=1,label='occluded')

    # target location / occlusion end
    ax.plot(end_time,5.5,'v',color=colors[speed_idx],markersize=8,zorder=500,markeredgecolor='k')

    ax.set_title(f'{speeds[speed_idx]:.2f} dva/sec')
    
    ax.set_ylim(-12,12)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.legend(frameon=False, fontsize=9)

fig.supxlabel('Time (s)')
fig.supylabel('Eye position (dva)')

plt.tight_layout()

plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/Group_eyetracking_transparent-vs-occluded.png')


In [7]:
# find when transparent and occluded deviate from true trajectory

bf_lists = [bf_transparent, bf_occluded]
diff_lists = [val_trans, val_occ]
bf_names = ['transparent', 'occluded']

first_positive_bf_times = {}

# find the first timepoint when there is evidence for a positive difference between the true trajectory and the gaze position for transparent and occluded

for name, bf_list, diff_list in zip(bf_names, bf_lists, diff_lists):

    temp_times = []

    for bf, diff, times in zip(bf_list, diff_list, bf_times):

        bf = np.asarray(bf)
        diff = np.asarray(diff)
        times = np.asarray(times)

        # BF > 6 AND difference is positive
        inds = np.where((bf > 6) & (diff > 0))[0]

        if len(inds) > 0:
            temp_times.append(times[inds[0]])
        else:
            temp_times.append(np.nan)

    first_positive_bf_times[name] = temp_times

    print('from trial onset',name, ':', temp_times)

# these proide timepoints after trial onset - to get timepoints after occlusion onset, subtract the occlusion start time for each speed:

first_positive_bf_times_relative = {}

for name in bf_names:

    relative_times = []

    for first_time, times in zip(first_positive_bf_times[name], bf_times):

        occlusion_onset = np.asarray(times)[0]

        if np.isfinite(first_time):
            relative_times.append(first_time - occlusion_onset)
        else:
            relative_times.append(np.nan)

    first_positive_bf_times_relative[name] = relative_times

    print('from occlusion onset',name, ':', relative_times)





from trial onset transparent : [np.float64(2.022), np.float64(1.616), np.float64(1.282), np.float64(1.064), np.float64(0.94), np.float64(0.617), np.float64(0.689), np.float64(0.484), np.float64(0.451), np.float64(0.417), np.float64(0.384), np.float64(0.367)]
from trial onset occluded : [nan, np.float64(1.482), np.float64(1.206), np.float64(1.002), np.float64(0.864), np.float64(0.802), np.float64(0.7), np.float64(0.664), np.float64(0.646), np.float64(0.657), np.float64(0.651), nan]
from occlusion onset transparent : [np.float64(0.4049999999999998), np.float64(0.399), np.float64(0.31500000000000006), np.float64(0.2470000000000001), np.float64(0.239), np.float64(0.0), np.float64(0.1379999999999999), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
from occlusion onset occluded : [nan, np.float64(0.2649999999999999), np.float64(0.239), np.float64(0.18500000000000005), np.float64(0.16300000000000003), np.float64(0.18500000000000005), np.float64(0.14899999

In [8]:
#plot the bayes factors for eye gaze deviation during occlusion for each of the 9 conditions 

#change "analysis" and "y_bf" to access  

#bf_transparent
#bf_occluded
#bf_trans_occ

plt.close('all')
fig,axs = plt.subplots(12,1,num=2,figsize=(a4_width, a4_height*0.7))

analysis = 'bf_transparent'
y_bf = bf_transparent

exponential_minmax=12
# x_occ_lab=np.arange(1, len(y_new)+1)

# Labels for the subplots
labels = ['(slow)', '', '', '', '', '', '', '','', '', '','(fast)']
for i, (ax, label) in enumerate(zip(axs.flat, labels)):

    #plot bfs
    plot_bfs(fig,ax,x=bf_times[i],y=y_bf[i],n_cols = 500,bf_cmap="coolwarm",exponential_minmax=exponential_minmax)

    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1
    fig.text(x, y, label, fontsize=10, fontweight='bold', va='top', ha='right')

    [ax.spines[i].set_visible(False) for i in ['top','right']]
    ax.tick_params(axis='y', which='major', labelsize=8)

    # # Only show x-axis labels on the final subplot
    # if i == len(axs) - 1:
    #     ax.set_xticks(np.arange(0, len(bf_times[])+1, 49))
    #     ax.set_xlabel('time since occlusion start (ms)')

    # else:
    ax.set_xticks([])

    
    # Only add y-axis label on middle graph
    if i == 4:
        ax.set_ylabel('BF',fontsize=10)

plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/Group_eyetracking_{analysis}.png')


In [9]:
plt.close('all')
fig,axs = plt.subplots(4,2,num=1,figsize=(a4_width, a4_height*0.9))


epos = all_epochs[0].copy()
x_pos = epos.metadata.occlusion_start_time.unique()[0]

# Labels for the subplots
labels = ['(a)', '(b)', '(c)','(d)','(e)','(f)','(g)','(h)']
for i, (ax, label) in enumerate(zip(axs.flat, labels)):
    pos = ax.get_position()  
    x = pos.x0 - 0.05
    y = pos.y1 + 0.02 
    fig.text(x, y, label, fontsize=12, fontweight='bold', va='top', ha='right')


#************************************
### (A) TRIAL
#************************************
ax = shrink_axes(axs[0][0],0.95,0.8)

summary_dat = epos.metadata.loc[epos.metadata.direction==1,:].groupby('velocity').mean(numeric_only=True)
for i in range(len(summary_dat)):
    traj = np.linspace(summary_dat['start_pos_deg'].iloc[i],np.abs(summary_dat['start_pos_deg'].iloc[i]),int(summary_dat['n_frames'].iloc[i]))
    # traj[traj>np.abs(summary_dat['occluder_start_pos_deg'].iloc[i])]=np.nan
    traj_tv = np.linspace(0,summary_dat['total_time'].iloc[i],int(summary_dat['n_frames'].iloc[i]))


    ax.plot(traj_tv[(traj>=-4.5)&(traj<=4.5)],traj[(traj>=-4.5)&(traj<=4.5)],':',color=colors[i])
    ax.plot(traj_tv[(traj<-4.5)],traj[(traj<-4.5)],'-',color=colors[i],lw=1.5)



x_pos = 0.2
ax.vlines(x_pos,-np.max(epos.metadata.occluder_start_pos_deg),np.max(epos.metadata.occluder_start_pos_deg),'k',linestyles='-',lw=2)
ax.hlines(-np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.hlines(np.max(epos.metadata.occluder_start_pos_deg),x_pos,x_pos+0.1,'k',linestyles='-',lw=2)
ax.annotate('occluder', xy=(x_pos, 5),xytext=(x_pos, 10),ha='left', va='top', fontsize=9,arrowprops=dict(arrowstyle='->', color='k'),clip_on=False)

for speed_idx in range(len(speeds)):
    ax.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=5,zorder=500,markeredgecolor=None)


ax.set_ylim([-13,13])
ax.set_xlabel('time since trial start (s)')
ax.set_ylabel('x-position object (dva)')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)


# #************************************
# ### (B) BEHAVIOUR - error in degrees
# #************************************
ax = shrink_axes(axs[0][1],0.85,0.8)

[ax.plot(error_deg_occluded_all[i].index,error_deg_occluded_all[i],'k',alpha=.3,lw=.5,color='k') for i in range(20)]
[ax.plot(error_deg_transparent_all[i].index,error_deg_transparent_all[i],'k:',alpha=.3,lw=.5,color='k') for i in range(20)]
ax.plot(error_deg_occluded_all[i].index,np.mean(error_deg_occluded_all,axis=0),'k',alpha=1,lw=2,label='occluded')
for speed_idx in range(len(speeds)):
    ax.plot(error_deg_occluded_all[0].index[speed_idx],np.mean(error_deg_occluded_all,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
ax.plot(error_deg_transparent_all[i].index,np.mean(error_deg_transparent_all,axis=0),'k:',alpha=1,lw=2,label='transparent')
ax.legend(frameon=False,fontsize=9)

# ax.axhline(y=0, color='k', linestyle='-.', lw=1.5, alpha=1, zorder=0)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('velocity (dva/sec)')
ax.set_ylabel('error (dva)')

#averages reported in the text

ave_position=np.mean(error_deg_occluded_all)

ave_position_each=np.mean(error_deg_occluded_all,axis=0)

ave_position_trans=np.mean(error_deg_transparent_all)

ave_position_trans_each=np.mean(error_deg_transparent_all,axis=0)

# BF for each speed

error_deg_occluded_all_arr = np.array(error_deg_occluded_all)
bf_err_deg = calculate_bf(error_deg_occluded_all_arr)

error_deg_trans_all_arr = np.array(error_deg_transparent_all)
bf_err_deg_trans = calculate_bf(error_deg_trans_all_arr)

# #************************************
# ### (C) BEHAVIOUR - temporal error
# #************************************
ax = shrink_axes(axs[1][0],0.85,0.8)

[ax.plot(temp_error_occluded_all[i].index,temp_error_occluded_all[i],'k',alpha=.3,lw=.5,color='k') for i in range(20)]
[ax.plot(temp_error_transparent_all[i].index,temp_error_transparent_all[i],'k:',alpha=.3,lw=.5,color='k') for i in range(20)]
ax.plot(temp_error_occluded_all[i].index,np.mean(temp_error_occluded_all,axis=0),'k',alpha=1,lw=2,label='occluded')
for speed_idx in range(len(speeds)):
    ax.plot(temp_error_occluded_all[0].index[speed_idx],np.mean(temp_error_occluded_all,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
ax.plot(temp_error_transparent_all[i].index,np.mean(temp_error_transparent_all,axis=0),'k:',alpha=1,lw=2,label='transparent')
ax.legend(frameon=False,fontsize=9)

# ax.axhline(y=0, color='k', linestyle='-.', lw=1.5, alpha=1, zorder=0)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('velocity (dva/sec)')
ax.set_ylabel('error (sec)')

# #************************************
# ### (D) BEHAVIOUR - subjective velocity
# #************************************
ax = shrink_axes(axs[1][1],0.85,0.8)

subjVelo = []
subjVelo_trans = []
ideal = []

for i in range(20):
    epos = all_epochs[i].copy()

    error_MP = (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time-epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).rt)
    error_trans_MP = (epos.metadata.loc[(epos.metadata.condition=='transparent'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time-epos.metadata.loc[(epos.metadata.condition=='transparent'),:].groupby('velocity').mean(numeric_only=True).rt)

    occ_duration = (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time) - (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_start_time)

    velo = np.sort(epos.metadata.velocity.unique())

    subjVelo.append(velo*(occ_duration/(occ_duration-error_MP))) #in seconds 
    subjVelo_trans.append(velo*(occ_duration/(occ_duration-error_trans_MP))) #in seconds 

    ideal.append(velo) 

[ax.plot(subjVelo[i].index,subjVelo[i],'k',alpha=.3,lw=.5,color='k') for i in range(20)]
[ax.plot(subjVelo_trans[i].index,subjVelo_trans[i],'k:',alpha=.3,lw=.5,color='k') for i in range(20)]
ax.plot(subjVelo[i].index,np.mean(subjVelo,axis=0),'k',alpha=1,lw=2,label='occluded')
ax.plot(subjVelo_trans[i].index,np.mean(subjVelo_trans,axis=0),'k:',alpha=1,lw=2,label='transparent')
for speed_idx in range(len(speeds)):
    ax.plot(subjVelo[0].index[speed_idx],np.mean(subjVelo,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
ax.legend(frameon=False,fontsize=9)

# Make x and y axes have the same limits
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),max(ax.get_xlim()[1], ax.get_ylim()[1])]

ax.set_xlim(lims)
ax.set_ylim(lims)

# Dotted y = x line
ax.plot(lims, lims, 'k-.', lw=1.5, zorder=0, label='perfect velocity estimate')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('velocity (dva/sec)')
ax.set_ylabel('subjective velocity')


# #************************************
# ### (E) EYETRACKING -- individual Sub-07 
# #************************************
sample_p = 6
ax = shrink_axes(axs[2][0],1,1)
[ax.spines[loc].set_visible(False) for loc in ['top','bottom','right','left']]
ax.set_xticks([])
ax.set_yticks([])

ax.set_ylabel('x-gaze position (dva)',labelpad=15)
ax.set_xlabel('time since trial start (s)',labelpad=8)

bottoms = [0.55,0.025]
condition_labels = [f'(i) sub-S{str(sample_p+1).zfill(2)}: occluded',f'(ii) sub-S{str(sample_p+1).zfill(2)}: transparent']
for condition_i,condition in enumerate(['occluded','transparent']):
    ax_in = make_subplots(ax,bottom=bottoms[condition_i],height="45%")
    x_ind = np.where([i=='xpos' for i in all_epochs[0].info.ch_names])[0][0]

    epos = all_epochs[sample_p].copy()
    epos = epos[epos.metadata.condition==condition]
    x_idx = np.where([i=='xpos' for i in epos.ch_names])[0][0]
    epos._data[epos.metadata.direction==2,x_idx,:] = epos._data[epos.metadata.direction==2,x_idx,:]*-1

    speeds = np.sort(epos.metadata.velocity.unique())


    for speed_idx in range(len(speeds)):
        end_time = epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0]
        tv = epos.times.copy()
        tv[tv>end_time]=np.nan
        trials = epos._data[epos.metadata.velocity==speeds[speed_idx],x_idx,:]
        trials[:,tv>end_time]=np.nan

        [ax_in.plot(tv,trial,color=colors[speed_idx],lw=0.2,alpha=.5) for trial in trials]
        ax_in.plot(tv,np.nanmean(trials,axis=0),color=colors[speed_idx],lw=0.8,alpha=1)

        ax_in.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=5,zorder=500,markeredgecolor=None)


    ax_in.spines['top'].set_visible(False)
    ax_in.spines['right'].set_visible(False)

    if condition_i == 0:
        ax_in.set_xticklabels('')
    ax_in.text(0.01, .98, condition_labels[condition_i], transform=ax_in.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='left')
    ax_in.set_ylim([-13,13])


# #************************************
# ### (F) EYETRACKING -- group
# #************************************
ax = shrink_axes(axs[2][1],1,1)
[ax.spines[loc].set_visible(False) for loc in ['top','bottom','right','left']]
ax.set_xticks([])
ax.set_yticks([])

ax.set_ylabel('x-gaze position (dva)',labelpad=15)
ax.set_xlabel('time since trial start (s)',labelpad=8)

bottoms = [0.55,0.025]
condition_labels = [f'(i) Average (n=20): occluded',f'(ii) Average (n=20): transparent']

for condition_i, condition in enumerate(['occluded', 'transparent']):
    ax_in = make_subplots(ax, bottom=bottoms[condition_i], height="45%")
    x_ind = np.where([i == 'xpos' for i in all_epochs[0].info.ch_names])[0][0]

    # correct trajectories (like in a)
    for i in range(len(summary_dat)):
        traj = np.linspace(summary_dat['start_pos_deg'].iloc[i],np.abs(summary_dat['start_pos_deg'].iloc[i]),int(summary_dat['n_frames'].iloc[i]))
        traj_tv = np.linspace(0,summary_dat['total_time'].iloc[i],int(summary_dat['n_frames'].iloc[i]))
        ax_in.plot(traj_tv[(traj>=-4.5)&(traj<=4.5)],traj[(traj>=-4.5)&(traj<=4.5)],':',color=colors[i],lw=0.5)
        ax_in.plot(traj_tv[(traj<-4.5)],traj[(traj<-4.5)],'-',color=colors[i],lw=0.5)

    # eyetracking results
    speeds = np.sort(all_epochs[0].metadata.velocity.unique())  # get speeds from first participant

    all_gaze_end = []
    all_end_dva = []

    for speed_idx in range(len(speeds)):
        all_participant_means = []

        for epos in all_epochs:  # loop over participants instead of sample_p
            epos = epos.copy()
            epos = epos[epos.metadata.condition == condition]
            x_idx = np.where([i == 'xpos' for i in epos.ch_names])[0][0]
            epos._data[epos.metadata.direction == 2, x_idx, :] = epos._data[epos.metadata.direction == 2, x_idx, :] * -1

            end_time = epos.metadata.loc[epos.metadata.velocity == speeds[speed_idx], 'occlusion_end_time'].unique()[0]
            tv = epos.times.copy()
            tv[tv > end_time] = np.nan
            trials = epos._data[epos.metadata.velocity == speeds[speed_idx], x_idx, :]
            trials[:, tv > end_time] = np.nan

            all_participant_means.append(np.nanmean(trials, axis=0))  # store per-participant mean

        grand_mean = np.nanmean(all_participant_means, axis=0)  # average across participants

        plot_mask = tv <= end_time #use the mask created by the nans in tv to extract final gaze position in grand_mean
        all_gaze_end.append(grand_mean[plot_mask][-1])
        all_end_dva.append(traj[(traj >= -4.5) & (traj <= 4.5)][-1])

        # plot individual participant means as faint lines (replaces single-trial lines)
        # [ax_in.plot(tv, p_mean, color=colors[speed_idx], lw=0.2, alpha=.5) for p_mean in all_participant_means]
        ax_in.plot(tv, grand_mean, color=colors[speed_idx], lw=0.8, alpha=1)
        ax_in.plot(epos.metadata.loc[epos.metadata.velocity==speeds[speed_idx],'occlusion_end_time'].unique()[0],5.5,'v',color=colors[speed_idx],markersize=5,zorder=500,markeredgecolor=None)


    ax_in.spines['top'].set_visible(False)
    ax_in.spines['right'].set_visible(False)
    if condition_i == 0:
        ax_in.set_xticklabels('')
    ax_in.text(0.01, .98, condition_labels[condition_i], transform=ax_in.transAxes, fontsize=9, verticalalignment='top', horizontalalignment='left')
    ax_in.set_ylim([-13, 13])

    #find difference between end gaze position and true TTC position
    gaze_diff = np.array(all_end_dva) - np.array(all_gaze_end)    
    ave_gaze_diff = np.nanmean(gaze_diff) 

    print(f"{condition}: Average gaze difference = {ave_gaze_diff:.2f}")

    



# #************************************
# ### (G) & (H) CORRELATIONS 
# #************************************
condition_labels = [' occluded',' transparent']
for condition_i,condition in enumerate(['occluded','transparent']):
    corrs = trialwise_df.loc[trialwise_df.condition==condition,:].groupby('subject').apply(lambda x: x['gaze_spatial_error'].corr(x['behavioral_spatial_error']))

    ax = shrink_axes(axs[3][condition_i],.9,.95)
    for s in range(1,21):
        x = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s),'gaze_spatial_error']
        y = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s),'behavioral_spatial_error']

        ax.plot(x,y,'.',color=subj_colors[s],alpha=0.4,markeredgecolor=None)

    
    ax.set_xlabel('error gaze position at occlusion end (dva)')
    if condition_i==0:
        ax.set_ylabel('error behavior (dva)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f'{condition_labels[condition_i]}:\n r = {np.mean(corrs):.2f}',loc='left',y=0.8,fontsize=9)

    ax.set_xlim([-5,15])
    ax.set_ylim([-5,40])

    ### inset for distributions of correlations
    axins = inset_axes(ax, width="30%", height="35%", loc="upper right")

    vplot=axins.violinplot(corrs, showmeans=True)

    for pc in vplot['bodies']:
        pc.set_facecolor('black')
        pc.set_edgecolor('black')
    for partname in ('cmeans', 'cmaxes', 'cmins', 'cbars'):
        if partname in vplot:
            vplot[partname].set_color('black')
            vplot[partname].set_linewidth(1.5)


    [axins.plot((s/75-10/75)+1,corrs.to_numpy()[s],'.',color=subj_colors[s],alpha=0.6) for s in range(20)]

    # axins.yaxis.tick_right()
    axins.yaxis.set_label_position("right")
    axins.set_xticks([])
    axins.set_ylabel('Correlation',fontsize=7)
    axins.set_ylim([-0.7,0.7])

    axins.spines['top'].set_visible(False)
    axins.spines['right'].set_visible(False)


    bf_out = calculate_bf(np.expand_dims(corrs,axis=1))
    print(f'mean correlation behaviour and gaze error for condition {condition} = {np.mean(corrs)}, BF = 10^{np.log10(np.array(bf_out))}')




# #************************************
# ### SAVE
# #************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3.pdf')


/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:57: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k" (-> color=(0.0, 0.0, 0.0, 1)). The keyword argument will take precedence.
  [ax.plot(error_deg_occluded_all[i].index,error_deg_occluded_all[i],'k',alpha=.3,lw=.5,color='k') for i in range(20)]
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:58: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "k:" (-> color='k'). The keyword argument will take precedence.
  [ax.plot(error_deg_transparent_all[i].index,error_deg_transparent_all[i],'k:',alpha=.3,lw=.5,color='k') for i in range(20)]
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:61: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "kv" (-> color='k'). The keyword argument will take precedence.
  ax.plot(error_deg_o

occluded: Average gaze difference = 0.97


/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:254: RuntimeWarning: Mean of empty slice
  all_participant_means.append(np.nanmean(trials, axis=0))  # store per-participant mean
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:254: RuntimeWarning: Mean of empty slice
  all_participant_means.append(np.nanmean(trials, axis=0))  # store per-participant mean
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:254: RuntimeWarning: Mean of empty slice
  all_participant_means.append(np.nanmean(trials, axis=0))  # store per-participant mean
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:254: RuntimeWarning: Mean of empty slice
  all_participant_means.append(np.nanmean(trials, axis=0))  # store per-participant mean
/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/1744619571.py:254: RuntimeWarning: Mean of empty slice
  all_participant_means.append(np.nanmean(tr

transparent: Average gaze difference = -0.08
mean correlation behaviour and gaze error for condition occluded = 0.19167769102675783, BF = 10^[1.8629536]
mean correlation behaviour and gaze error for condition transparent = -0.02043822898290587, BF = 10^[-0.5899356]


In [10]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

bf_matrix = np.full((12, 12), np.nan)

for col1 in range(12):
    for col2 in range(col1 + 1, 12):

        difference = (error_deg_occluded_all_arr[:, col1] - error_deg_occluded_all_arr[:, col2]).reshape(-1, 1)

        bf = calculate_bf(difference)[0]

        bf_matrix[col1, col2] = bf
        bf_matrix[col2, col1] = bf


bf_plot = bf_matrix.copy()
# Keep only lower triangle
bf_plot[np.triu_indices_from(bf_plot)] = np.nan

# Create a custom blue -> gray -> red colormap
blue = (0.251, 0.424, 0.851)
red  = (0.859, 0.239, 0.239)

cmap = LinearSegmentedColormap.from_list('bf_cmap',[(0.00, blue),(0.50, 'white'),(1.00, red)])

# Use log10 BF so that BF and its reciprocal are symmetric

# # Limits:
# # 1/6 -> blue
# # 1    -> white
# # 6    -> red

log_bf = np.log10(bf_plot)

log_bf_clipped = np.clip(log_bf,np.log10(1/6),np.log10(6))

norm = TwoSlopeNorm(vmin=np.log10(1/6),vcenter=0,vmax=np.log10(6))

# Plot

fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(log_bf_clipped,cmap=cmap,norm=norm,aspect='equal')

# Add BF values to cells

for row in range(12):
    for col in range(12):

        bf = bf_matrix[row, col]

        # Only show lower triangle
        if row > col and not np.isnan(bf):
            ax.text(col,row,f'{bf:.2f}',ha='center',va='center',fontsize=9)

ax.set_xlabel('Speed', fontsize=14)
ax.set_ylabel('Speed', fontsize=14)

ax.set_xticks(range(12))
ax.set_yticks(range(12))

ax.set_xticklabels(range(1, 13))
ax.set_yticklabels(range(1, 13))

cbar = fig.colorbar(im, ax=ax)

# Tick locations expressed as log10(BF)
cbar.set_ticks([np.log10(1/6), np.log10(1),
    np.log10(6)])

cbar.set_ticklabels([r'$BF_{10}<1/6$',r'$BF_{10}=1$',r'$BF_{10}>6$'])

cbar.set_label(r'$BF_{10}$', fontsize=14)

ax.set_title(r'Pairwise Bayes Factors - Occluded($BF_{10}$)',fontsize=16)

plt.tight_layout()
plt.show()

# #************************************
# ### SAVE
# #************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3_bf_beh_occ_supp.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3_bf_beh_occ_supp.pdf')

In [11]:
# BF between speeds for transparent

bf_trans_matrix = np.full((12, 12), np.nan)

for col1 in range(12):
    for col2 in range(col1 + 1, 12):

        difference = (error_deg_trans_all_arr[:, col1] - error_deg_trans_all_arr[:, col2]).reshape(-1, 1)

        bf = calculate_bf(difference)[0]

        bf_trans_matrix[col1, col2] = bf
        bf_trans_matrix[col2, col1] = bf

bf_trans_plot = bf_trans_matrix.copy()

# Keep only lower triangle
bf_trans_plot[np.triu_indices_from(bf_trans_plot)] = np.nan

# Use log10 BF so that BF and its reciprocal are symmetric
log_bf_trans = np.log10(bf_trans_plot)

log_bf_trans_clipped = np.clip(log_bf_trans,np.log10(1/6),np.log10(6))

norm = TwoSlopeNorm(vmin=np.log10(1/6),vcenter=0,vmax=np.log10(6))

fig, ax = plt.subplots(figsize=(9, 8))

im = ax.imshow(log_bf_trans_clipped,cmap=cmap,norm=norm,aspect='equal')

for row in range(12):
    for col in range(12):

        bf_trans = bf_trans_matrix[row, col]

        # Only show lower triangle
        if row > col and not np.isnan(bf_trans):
            ax.text(col,row,f'{bf_trans:.2f}',ha='center',va='center',fontsize=9)

ax.set_xlabel('Speed', fontsize=14)
ax.set_ylabel('Speed', fontsize=14)

ax.set_xticks(range(12))
ax.set_yticks(range(12))

ax.set_xticklabels(range(1, 13))
ax.set_yticklabels(range(1, 13))

cbar = fig.colorbar(im, ax=ax)

# Tick locations expressed as log10(BF)
cbar.set_ticks([np.log10(1/6),np.log10(1),np.log10(6)])

cbar.set_ticklabels([r'$BF_{10}<1/6$',r'$BF_{10}=1$',r'$BF_{10}>6$'])

cbar.set_label(r'$BF_{10}$', fontsize=14)

ax.set_title(r'Pairwise Bayes Factors - Transparent ($BF_{10}$)',fontsize=16)

plt.tight_layout()
plt.show()

# #************************************
# ### SAVE
# #************************************
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3_bf_beh_trans_supp.png',dpi=300)
plt.savefig(f'{bids_dir}/{task}/derivatives/figures/fig_exp3_bf_beh_trans_supp.pdf')

In [12]:
# Correlation per speed

fig,axs = plt.subplots(6,2,figsize=(6,13))

condition_labels = [' occluded',' transparent']
for ax_curr,vel in zip(axs.flatten(), np.sort(trialwise_df.velocity.unique())):
    condition_i = 0
    condition = 'occluded'
    corrs = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.velocity==vel),:].groupby('subject').apply(lambda x: x['gaze_spatial_error'].corr(x['behavioral_spatial_error']))

    corrs_all = np.array(corrs) #removed any empty data rows... ok?
    corrs_all = corrs_all[~np.isnan(corrs_all)]
    corrs_all = corrs_all.reshape(-1, 1)

    bf_out = calculate_bf(corrs_all)  
    print(f'mean correlation behaviour and gaze error  = {np.mean(corrs):.2f}, BF = 10^{np.log10(np.array(bf_out))}')            

    ax = shrink_axes(ax_curr,.9,.95)
    for s in range(1,21):
        x = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s)&(trialwise_df.velocity==vel),'gaze_spatial_error']
        y = trialwise_df.loc[(trialwise_df.condition==condition)&(trialwise_df.subject==s)&(trialwise_df.velocity==vel),'behavioral_spatial_error']

        ax.plot(x,y,'.',color=subj_colors[s],alpha=0.4,markeredgecolor=None)

    

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_title(f'{condition_labels[condition_i]}, velocity {str(np.round(vel,2))} deg/sec:\n r = {np.mean(corrs):.2f}',loc='left',y=0.8,fontsize=9)

    ax.set_xlim([-5,15])
    ax.set_ylim([-5,40])

fig.supxlabel('error gaze position at occlusion end (dva)')
fig.supylabel('error behavior (dva)')
plt.tight_layout()


plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/correlation_per_speed.png')
plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/correlation_per_speed.pdf')



mean correlation behaviour and gaze error  = 0.46, BF = 10^[5.77587268]
mean correlation behaviour and gaze error  = 0.44, BF = 10^[2.64656389]
mean correlation behaviour and gaze error  = 0.40, BF = 10^[1.85285888]
mean correlation behaviour and gaze error  = 0.21, BF = 10^[0.77251153]


/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


mean correlation behaviour and gaze error  = 0.21, BF = 10^[0.64159888]
mean correlation behaviour and gaze error  = 0.19, BF = 10^[0.05008018]
mean correlation behaviour and gaze error  = 0.18, BF = 10^[-0.09685166]
mean correlation behaviour and gaze error  = 0.14, BF = 10^[-0.21710428]
mean correlation behaviour and gaze error  = 0.16, BF = 10^[-0.12274189]
mean correlation behaviour and gaze error  = 0.18, BF = 10^[-0.18512309]


/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


mean correlation behaviour and gaze error  = 0.17, BF = 10^[0.0682748]


/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/edwardsgc/miniforge3/envs/py-r-bayes/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


mean correlation behaviour and gaze error  = 0.15, BF = 10^[-0.16440473]


In [13]:
# Battaglini et al. 2013 estimate the subjective velocity during occlusion (what they call "remembered speed")

#remembered speed = length of invisible trajectory/(RT - time of target disappearance)

#S3 and S11 seem to have negative values - both these subjects show predictive behavior in our prior calculations

rem_speed = []
ideal = []

for i in range(20):
    epos = all_epochs[i].copy()

    len_invis_traj = (
        (epos.metadata.loc[(epos.metadata.condition=='occluded'),:]
        .groupby('velocity').mean(numeric_only=True).occlusion_end_time)
        -
        (epos.metadata.loc[(epos.metadata.condition=='occluded'),:]
        .groupby('velocity').mean(numeric_only=True).occlusion_start_time)
    ) * np.sort(epos.metadata.loc[(epos.metadata.condition=='occluded'),'velocity'].unique())
    
    rt = epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).rt

    t_tar_dis = (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_start_time)

    rem_speed.append(len_invis_traj/(rt-t_tar_dis)) #in seconds 
    ideal.append(len_invis_traj) 

fig, ax = plt.subplots(figsize=(9, 8))

# [ax.plot(rem_speed[i].index,rem_speed[i],'k',alpha=.3,lw=.5,color='k') for i in range(20)]
ax.plot(rem_speed[i].index,np.mean(rem_speed,axis=0),'k',alpha=1,lw=2,label='occluded')
for speed_idx in range(len(speeds)):
    ax.plot(rem_speed[0].index[speed_idx],np.mean(rem_speed,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
ax.legend(frameon=False,fontsize=9)

# Make x and y axes have the same limits
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1])
]

ax.set_xlim(lims)
ax.set_ylim(lims)

# Dotted y = x line
ax.plot(lims, lims, 'k:', lw=1.5, zorder=0)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('velocity (dva/sec)')
ax.set_ylabel('subjective velocity')


/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/2864660449.py:33: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "kv" (-> color='k'). The keyword argument will take precedence.
  ax.plot(rem_speed[0].index[speed_idx],np.mean(rem_speed,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])


Text(0, 0.5, 'subjective velocity')

In [14]:
# Makin & Poliakoff estimate subjective velocity across whole trial:

#error: time of would be reappearance - response time (# note it has to be occlusion_end_time - rt!!)
#subjective velocity = actual velocity * (occlusion duration/occlusion duration-error)

# Battaglini et al. 2013 estimate the subjective velocity during occlusion (what they call "remembered speed") - it produced the same result

subjVelo = []
ideal = []

for i in range(20):
    epos = all_epochs[i].copy()

    error_MP = (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time-epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).rt)

    occ_duration = (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_end_time) - (epos.metadata.loc[(epos.metadata.condition=='occluded'),:].groupby('velocity').mean(numeric_only=True).occlusion_start_time)

    velo = np.sort(epos.metadata.velocity.unique())

    subjVelo.append(velo*(occ_duration/(occ_duration-error_MP))) #in seconds 
    ideal.append(velo) 

fig, ax = plt.subplots(figsize=(9, 8))

ax.plot(subjVelo[i].index,np.mean(subjVelo,axis=0),'k',alpha=1,lw=2,label='occluded')
for speed_idx in range(len(speeds)):
    ax.plot(subjVelo[0].index[speed_idx],np.mean(subjVelo,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
ax.legend(frameon=False,fontsize=9)

# Make x and y axes have the same limits
lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1])
]

ax.set_xlim(lims)
ax.set_ylim(lims)

# Dotted y = x line
ax.plot(lims, lims, 'k:', lw=1.5, zorder=0)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('velocity (dva/sec)')
ax.set_ylabel('subjective velocity')

plt.savefig(f'/Users/edwardsgc/Library/CloudStorage/OneDrive-NationalInstitutesofHealth/OcclusionPaper/data/exp3/derivatives/figures/SubjVelo.png')


/var/folders/bh/8mpq1ky171bccqq2bzyhg4ynwp42l4/T/ipykernel_57905/3978195112.py:27: UserWarning: color is redundantly defined by the 'color' keyword argument and the fmt string "kv" (-> color='k'). The keyword argument will take precedence.
  ax.plot(subjVelo[0].index[speed_idx],np.mean(subjVelo,axis=0)[speed_idx],'kv',alpha=1,lw=2,markersize=6,color=colors[speed_idx])
